In [1]:
import joblib

X_train_scaled = joblib.load("../data/processed/X_train_scaled.pkl")
X_test_scaled = joblib.load("../data/processed/X_test_scaled.pkl")
y_train = joblib.load("../data/processed/y_train.pkl")
y_test = joblib.load("../data/processed/y_test.pkl")

print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)

X_train: (100778, 119)
X_test: (25195, 119)


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_scaled, y_train)

print("Best params:", random_search.best_params_)
print("Best CV F1 score:", random_search.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 30}
Best CV F1 score: 0.998538094657113


In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

best_rf = random_search.best_estimator_

y_pred = best_rf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.999047430045644
Precision: 0.9994877923851802
Recall: 0.9984649496844619
F1 Score: 0.9989761092150171
ROC-AUC: 0.9990097411574733

Confusion Matrix:
 [[13463     6]
 [   18 11708]]


In [4]:
import pandas as pd
import numpy as np

columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

df_holdout = pd.read_csv("../data/KDDTest+.txt", names=columns)
df_holdout["binary_label"] = df_holdout["label"].apply(lambda x: "normal" if x == "normal" else "attack")
df_holdout = df_holdout.drop(columns=["label", "difficulty"])

for col in ["duration", "src_bytes", "dst_bytes"]:
    df_holdout[col] = np.log1p(df_holdout[col])

df_holdout = pd.get_dummies(df_holdout, columns=["protocol_type", "service", "flag"], drop_first=True)
df_holdout["binary_label"] = df_holdout["binary_label"].map({"normal": 0, "attack": 1})

print("Holdout shape:", df_holdout.shape)

Holdout shape: (22544, 114)


In [5]:
# Get the training feature columns (excluding the target)
train_columns = X_train.columns  # if X_train isn't in this notebook, use joblib to reload it, see note below

# Separate holdout features and target
X_holdout = df_holdout.drop(columns=["binary_label"])
y_holdout = df_holdout["binary_label"]

# Align: add any missing columns (fill with 0), drop any extra columns, reorder to match training
X_holdout = X_holdout.reindex(columns=train_columns, fill_value=0)

print("Aligned holdout shape:", X_holdout.shape)

NameError: name 'X_train' is not defined

In [6]:
# Reload the original processed dataframe to get the training column names
df_train_full = pd.read_csv("../data/processed_train.csv")
train_columns = df_train_full.drop(columns=["binary_label"]).columns

# Separate holdout features and target
X_holdout = df_holdout.drop(columns=["binary_label"])
y_holdout = df_holdout["binary_label"]

# Align: add any missing columns (fill with 0), drop any extra columns, reorder to match training
X_holdout = X_holdout.reindex(columns=train_columns, fill_value=0)

print("Aligned holdout shape:", X_holdout.shape)

Aligned holdout shape: (22544, 119)


In [7]:
scaler = joblib.load("../models/scaler.pkl")

X_holdout_scaled = scaler.transform(X_holdout)

print("Scaled holdout shape:", X_holdout_scaled.shape)

Scaled holdout shape: (22544, 119)


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

y_holdout_pred = best_rf.predict(X_holdout_scaled)

print("Holdout Accuracy:", accuracy_score(y_holdout, y_holdout_pred))
print("Holdout Precision:", precision_score(y_holdout, y_holdout_pred))
print("Holdout Recall:", recall_score(y_holdout, y_holdout_pred))
print("Holdout F1 Score:", f1_score(y_holdout, y_holdout_pred))
print("Holdout ROC-AUC:", roc_auc_score(y_holdout, y_holdout_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_holdout, y_holdout_pred))

Holdout Accuracy: 0.7722232079489
Holdout Precision: 0.9683621319055732
Holdout Recall: 0.620120003116964
Holdout F1 Score: 0.756068595316137
Holdout ROC-AUC: 0.7966731207017217

Confusion Matrix:
 [[9451  260]
 [4875 7958]]


## Holdout Test Results (KDDTest+.txt)
- Accuracy dropped from ~99.9% (internal test) to 77.2% on the true holdout set.
- Recall dropped significantly (99.8% -> 62.0%) - the model misses many real attacks.
- Precision remained high (96.8%) - when it predicts "attack," it's usually correct.
- This gap is expected and well-documented for NSL-KDD: KDDTest+.txt intentionally 
  contains attack types and variants not present in KDDTrain+.txt, testing the 
  model's ability to generalize rather than memorize.
- This demonstrates the internal 99.9% score was partly a reflection of the training 
  distribution, not true real-world generalization - an important finding to report 
  honestly rather than only showcasing the higher number.

In [9]:
# Reload the raw holdout file again, just to recover the original attack-type labels
df_holdout_raw = pd.read_csv("../data/KDDTest+.txt", names=columns)

# Build a results dataframe aligned by row position
results_df = pd.DataFrame({
    "true_label": df_holdout_raw["label"].values,      # original 23-class label
    "binary_true": y_holdout.values,                    # 0 = normal, 1 = attack
    "binary_pred": y_holdout_pred                        # model's prediction
})

# Focus on real attacks the model completely missed (predicted "normal" instead)
missed = results_df[(results_df["binary_true"] == 1) & (results_df["binary_pred"] == 0)]

print("Total missed attacks:", len(missed))
print("\nMissed attacks by type:")
print(missed["true_label"].value_counts())

Total missed attacks: 4875

Missed attacks by type:
true_label
guess_passwd       1231
warezmaster         798
processtable        668
mscan               595
apache2             519
snmpguess           331
mailbomb            293
snmpgetattack       178
httptunnel          118
buffer_overflow      18
multihop             16
named                15
sendmail             14
ps                   13
rootkit              13
back                 10
xterm                10
xlock                 9
saint                 5
pod                   3
xsnoop                3
ftp_write             3
loadmodule            2
worm                  2
land                  2
neptune               2
perl                  1
sqlattack             1
imap                  1
phf                   1
Name: count, dtype: int64


In [10]:
attack_totals = results_df[results_df["binary_true"] == 1]["true_label"].value_counts()
missed_counts = missed["true_label"].value_counts()

miss_rate = (missed_counts / attack_totals * 100).sort_values(ascending=False)
print("Miss rate (%) by attack type:")
print(miss_rate)


Miss rate (%) by attack type:
true_label
xlock              100.000000
loadmodule         100.000000
snmpgetattack      100.000000
snmpguess          100.000000
worm               100.000000
sendmail           100.000000
mailbomb           100.000000
rootkit            100.000000
imap               100.000000
guess_passwd       100.000000
ftp_write          100.000000
processtable        97.518248
buffer_overflow     90.000000
multihop            88.888889
httptunnel          88.721805
named               88.235294
ps                  86.666667
warezmaster         84.533898
xterm               76.923077
xsnoop              75.000000
apache2             70.420624
mscan               59.738956
sqlattack           50.000000
perl                50.000000
phf                 50.000000
land                28.571429
pod                  7.317073
back                 2.785515
saint                1.567398
neptune              0.042946
ipsweep                   NaN
nmap                      NaN